# **GRU**

GRU — это тип рекуррентной нейросети (RNN), придуманный как более простой аналог LSTM, который умеет помнить важную информацию во времени и забывать лишнее.

Используется для:

временных рядов (цены, сигналы, сенсоры),

текста (NLP),

zₜ = σ(W_z·[hₜ₋₁, xₜ])
rₜ = σ(W_r·[hₜ₋₁, xₜ])

h̃ₜ = tanh(W·[rₜ ⊙ hₜ₋₁, xₜ])
hₜ = (1 − zₜ) ⊙ hₜ₋₁ + zₜ ⊙ h̃ₜ


zₜ — update gate

rₜ — reset gate

h̃ₜ — кандидат скрытого состояния

In [ ]:
import numpy as np
import pandas as pd
from preprocessing.preprocess import prep
from preprocessing.target import ttp_target
from metrics.Metrics import merged_metrics
name = "AFKS"
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

In [ ]:

from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv("Brent.csv")
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,AO_saucer_down,EntrySignal,EntryReason,Fractal_Up_conf,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct,AddOn_Ready,AddOn_Triggered
0,2015-10-26 10:00:00,48.05,48.12,47.89,48.09,48.28815,48.15758,48.06107,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
1,2015-10-26 11:00:00,48.10,48.36,48.00,48.30,48.26098,48.12601,48.05886,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
2,2015-10-26 12:00:00,48.30,48.35,48.18,48.30,48.23552,48.11588,48.05209,0,0,...,0,-1,three_color,0,0,NaN,NaN,NaN,0,0
3,2015-10-26 13:00:00,48.30,48.34,48.05,48.09,48.21971,48.10765,48.04267,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
4,2015-10-26 14:00:00,48.11,48.28,47.97,48.07,48.19550,48.09732,48.07014,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36239,2025-10-21 23:00:00,61.55,61.73,61.55,61.65,61.17612,61.14427,61.29423,0,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36240,2025-10-22 09:00:00,62.27,62.66,62.26,62.46,61.16565,61.21749,61.31439,1,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36241,2025-10-22 10:00:00,62.45,62.49,62.19,62.35,61.13752,61.23530,61.35151,0,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36242,2025-10-22 11:00:00,62.35,62.55,62.25,62.36,61.14002,61.25526,61.40921,0,0,...,0,0,NaN,1,0,61.52,1.0,0.3,0,0


In [ ]:
scale_cols = [
    "Open", "High", "Low", "Close",
    "Alligator_Jaw", "Alligator_Teeth", "Alligator_Lips",
    "AO",
    "AddOn_Anchor_Level", "AddOn_Size_Pct"
]

scaler = RobustScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])


In [ ]:
columns = [
    "AddOn_Anchor_Level",
    "AddOn_Anchor_IsUp",
    "AddOn_Size_Pct"
]

df = df.dropna(subset=columns).reset_index(drop=True)

In [ ]:
import numpy as np
import pandas as pd

# если ещё не загружал:
# df = pd.read_csv("Brent.csv")

H = 20  # горизонт, можно менять

def add_goodtrade_target(df, horizon=20):
    df = df.copy()

    # будущая цена
    df['Close_fwd'] = df['Close'].shift(-horizon)

    # доходность по направлению сигнала
    ret_long = (df['Close_fwd'] - df['Close']) / df['Close']
    ret_short = (df['Close'] - df['Close_fwd']) / df['Close']

    ret = np.where(
        df['EntrySignal'] > 0, ret_long,
        np.where(df['EntrySignal'] < 0, ret_short, 0.0)
    )

    df['ret_H'] = ret

    # таргет
    df['GoodTrade'] = ((df['EntrySignal'] != 0) & (df['ret_H'] > 0)).astype(int)

    # убираем хвост, где нет Close_fwd
    df = df.iloc[:-horizon].reset_index(drop=True)
    return df

df = add_goodtrade_target(df, horizon=H)

print("'GoodTrade' в колонках:", 'GoodTrade' in df.columns)
print(df[['Close', 'Close_fwd', 'ret_H', 'EntrySignal', 'GoodTrade']].head())


'GoodTrade' в колонках: True
      Close  Close_fwd     ret_H  EntrySignal  GoodTrade
0 -0.847430  -0.879768  0.000000            0          0
1 -0.856136  -0.878939  0.000000            0          0
2 -0.859453  -0.876451  0.000000            0          0
3 -0.863599  -0.876036 -0.014402           -1          0
4 -0.870232  -0.878939  0.000000            0          0


In [ ]:
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,Fractal_Up_conf,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct,AddOn_Ready,AddOn_Triggered,Close_fwd,ret_H,GoodTrade
0,2015-10-26 19:00:00,-0.851161,-0.846918,-0.847880,-0.847430,-0.838401,-0.838317,-0.842386,1,0,...,0,1,-0.826423,0.0,0.0,1,1,-0.879768,0.000000,0
1,2015-10-26 20:00:00,-0.847015,-0.850641,-0.854530,-0.856136,-0.838041,-0.839319,-0.845333,0,0,...,0,0,-0.826423,0.0,0.0,0,0,-0.878939,0.000000,0
2,2015-10-26 21:00:00,-0.855721,-0.854365,-0.855362,-0.859453,-0.837932,-0.841183,-0.846775,0,0,...,1,0,-0.826423,0.0,0.0,0,0,-0.876451,0.000000,0
3,2015-10-26 22:00:00,-0.859038,-0.863881,-0.859102,-0.863599,-0.838056,-0.843231,-0.847305,0,0,...,0,0,-0.826423,0.0,0.0,0,0,-0.876036,-0.014402,0
4,2015-10-26 23:00:00,-0.863184,-0.867191,-0.864090,-0.870232,-0.838778,-0.844450,-0.848769,0,0,...,0,0,-0.826423,0.0,0.0,0,0,-0.878939,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36210,2025-10-20 18:00:00,-0.313433,-0.307406,-0.315461,-0.319237,-0.290522,-0.294306,-0.299641,0,0,...,0,1,-0.182927,0.0,0.0,0,0,-0.275705,0.000000,0
36211,2025-10-20 19:00:00,-0.315506,-0.317749,-0.310474,-0.313847,-0.291317,-0.295975,-0.303799,0,0,...,0,0,-0.182927,0.0,0.0,0,0,-0.242123,0.000000,0
36212,2025-10-20 20:00:00,-0.313433,-0.306578,-0.309227,-0.303483,-0.291634,-0.297384,-0.305587,0,0,...,0,0,-0.182927,0.0,0.0,0,0,-0.246683,0.000000,0
36213,2025-10-20 21:00:00,-0.303068,-0.303269,-0.298421,-0.304312,-0.292167,-0.300306,-0.306851,1,0,...,0,0,-0.182927,0.0,0.0,0,0,-0.246269,0.000000,0


In [ ]:
# Берём только числовые колонки
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Что точно НЕ идёт в признаки
drop_feature_cols = ['Close_fwd', 'ret_H', 'GoodTrade']

feature_cols = [c for c in numeric_cols if c not in drop_feature_cols]
print("Числовые фичи для GRU:", feature_cols)

# Масштабируем признаки (для стабильного обучения нейросети)
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[feature_cols] = scaler.fit_transform(df_scaled[feature_cols])

print(df_scaled[feature_cols].head())


Числовые фичи для GRU: ['Open', 'High', 'Low', 'Close', 'Alligator_Jaw', 'Alligator_Teeth', 'Alligator_Lips', 'Fractal_Up', 'Fractal_Down', 'AO', 'Color AO', 'Alligator_Bullish', 'Alligator_Bearish', 'AlligatorStart_Long', 'AlligatorStart_Short', 'AO_sign', 'AO_zero_up', 'AO_zero_down', 'AO_three_green', 'AO_three_red', 'AO_saucer_up', 'AO_saucer_down', 'EntrySignal', 'Fractal_Up_conf', 'Fractal_Down_conf', 'AddOn_Anchor_Level', 'AddOn_Anchor_IsUp', 'AddOn_Size_Pct', 'AddOn_Ready', 'AddOn_Triggered']
       Open      High       Low     Close  Alligator_Jaw  Alligator_Teeth  \
0 -1.137332 -1.133541 -1.130863 -1.131572      -1.115856        -1.116862   
1 -1.131569 -1.138715 -1.140111 -1.143674      -1.115356        -1.118251   
2 -1.143671 -1.143889 -1.141267 -1.148284      -1.115205        -1.120837   
3 -1.148281 -1.157111 -1.146469 -1.154047      -1.115377        -1.123676   
4 -1.154044 -1.161710 -1.153405 -1.163268      -1.116379        -1.125368   

   Alligator_Lips  Fractal_Up  

In [ ]:
SEQ_LEN = 50  # длина "окна" в барах, можно 30, 50, 100

def make_sequences(df, feature_cols, seq_len=50):
    data = df[feature_cols].values
    targets = df['GoodTrade'].values
    signals = df['EntrySignal'].values

    X_list, y_list = [], []

    # начинаем с seq_len-1, чтобы слева хватило истории
    for i in range(seq_len - 1, len(df)):
        # нас интересуют только те точки, где есть сигнал
        if signals[i] == 0:
            continue

        X_seq = data[i - seq_len + 1 : i + 1, :]  # [seq_len, num_features]
        y_val = targets[i]                        # GoodTrade на последнем баре

        X_list.append(X_seq)
        y_list.append(y_val)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)

    return X, y

X_all, y_all = make_sequences(df_scaled, feature_cols, seq_len=SEQ_LEN)

print("Форма X_all:", X_all.shape)  # (num_samples, SEQ_LEN, num_features)
print("Форма y_all:", y_all.shape)
print("Доля GoodTrade=1:", y_all.mean())


Форма X_all: (36166, 50, 30)
Форма y_all: (36166,)
Доля GoodTrade=1: 0.024000442404468286


In [ ]:
split_idx = int(len(X_all) * 0.8)

X_train, X_test = X_all[:split_idx], X_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (28932, 50, 30) Test: (7234, 50, 30)


In [ ]:
num_features = X_train.shape[2]

model = Sequential([
    GRU(64, input_shape=(SEQ_LEN, num_features), return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # для бинарной классификации
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 38s 75ms/step - accuracy: 0.9416 - loss: 0.1989 - val_accuracy: 0.9747 - val_loss: 0.0396
Epoch 2/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.9753 - loss: 0.0470 - val_accuracy: 0.9754 - val_loss: 0.0347
Epoch 3/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 19s 41ms/step - accuracy: 0.9779 - loss: 0.0373 - val_accuracy: 0.9780 - val_loss: 0.0340
Epoch 4/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 18s 39ms/step - accuracy: 0.9754 - loss: 0.0383 - val_accuracy: 0.9773 - val_loss: 0.0340
Epoch 5/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 22s 42ms/step - accuracy: 0.9760 - loss: 0.0369 - val_accuracy: 0.9772 - val_loss: 0.0340
Epoch 6/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 18s 39ms/step - accuracy: 0.9773 - loss: 0.0341 - val_accuracy: 0.9768 - val_loss: 0.0340
Epoch 7/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 20s 39ms/step - accuracy: 0.9772 - loss: 0.0364 - val_accuracy: 0.9772 - val_loss: 0.0340
Epoch 8/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 19s 42ms/step - accuracy: 0.9778 - loss: 0

In [ ]:
y_proba = model.predict(X_test).ravel()
y_pred = (y_proba >= 0.5).astype(int)

print("GRU AUC:", roc_auc_score(y_test, y_proba))
print("\nОтчёт по классификации (GRU):")
print(classification_report(y_test, y_pred))
print("Матрица ошибок (GRU):")
print(confusion_matrix(y_test, y_pred))


227/227 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
GRU AUC: 0.9878280927319966

Отчёт по классификации (GRU):
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7064
           1       0.51      0.63      0.56       170

    accuracy                           0.98      7234
   macro avg       0.75      0.81      0.78      7234
weighted avg       0.98      0.98      0.98      7234

Матрица ошибок (GRU):
[[6962  102]
 [  63  107]]
